In [1]:
# ======================================================================
# 【模块一】基础数据体检
# ======================================================================
#
# [量化行业判断] 数据体检是因子研究的必要前置，不是可选步骤。
# 盲目计算因子而不先理解数据结构，会导致：
#   1. rolling 窗口的时序假设错误（数据未排序）
#   2. NaN 含义混淆（结构性缺失 vs 数据质量问题）
#   3. 因子分布失真（被上市前/退市后的占位行污染）
#
# [代码工程] 本模块只做探索，不修改 df 的值，只做类型转换和排序。
# 所有结果用 print 输出，方便在 Jupyter 中逐段运行。

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Heiti TC'

WINDOW_DAYS = 20
LAMBDA      = 0.2
SELECT_N    = int(WINDOW_DAYS * LAMBDA)   # = 4
FILE_PATH   = "数据.csv"

# ── 读取原始数据 ─────────────────────────────────────────────────
df = pd.read_csv(FILE_PATH)
df['date'] = pd.to_datetime(df['date'])
df_raw = df.sort_values(['order_book_id', 'date']).reset_index(drop=True)


# ── 1.1  date 转 datetime ─────────────────────────────────────────────
# [代码工程] pd.to_datetime 比字符串比较安全：
#   - 遇到非法日期字符串会抛异常（字符串比较会静默返回错误结果）
#   - 后续 .dt.date, .dt.year, .dt.month 等访问器依赖 datetime 格式
#   - rolling / resample / merge 的时序操作均依赖此格式
df['date'] = pd.to_datetime(df['date'])

# ── 1.2  按 (股票, 日期) 排序 ─────────────────────────────────────────
# [量化行业判断] rolling、shift 都依赖时间顺序。
# 如果数据未排序，rolling(20) 拿到的是行号相邻的20行，而不是时间相邻的20天。
# 这是最常见的因子计算隐患之一，在工业界是必须要求的前置步骤。
df = df.sort_values(['order_book_id', 'date']).reset_index(drop=True)

print("=" * 60)
print("【模块一】基础数据体检")
print("=" * 60)

# ── 1.3  基础维度 ─────────────────────────────────────────────────────
total_rows   = len(df)
total_stocks = df['order_book_id'].nunique()
total_dates  = df['date'].nunique()
date_min     = df['date'].min()
date_max     = df['date'].max()

print(f"\n  df.shape         : {df.shape}")
print(f"  df.columns       : {list(df.columns)}")
print()

# ── 1.4  字段类型检查 ────────────────────────────────────────────────
# [代码工程] dtypes 异常是数据读取失败的常见表现：
#   - buy_value 读成 object 通常说明原始 csv 里有逗号或引号
#   - date 读成 object 说明未完成 datetime 转换
print("  字段类型：")
for col, dtype in df.dtypes.items():
    print(f"    {col:<20s}  {dtype}")
print()

print(f"  总行数           : {total_rows:,}")
print(f"  独立股票数       : {total_stocks:,} 只")
print(f"  独立交易日数     : {total_dates:,} 天")
print(f"  日期范围         : {date_min.date()}  →  {date_max.date()}")
print()

# ── 1.5  面板完整性检验（最关键的结构性判断）────────────────────────
# [量化行业判断] 这里区分两种完全不同的数据结构：
#
#   情况A：行不存在（panel 有缺口）
#     总行数 < 股票数 × 交易日数
#     新股上市前、退市后的行不存在。
#     rolling(20) 会跨越"行不存在"的缺口，把上市前和上市后的数据错误连接。
#     必须先 reindex 补行，再做因子计算。
#
#   情况B：行存在但值为 NaN（矩形面板，已补全）
#     总行数 = 股票数 × 交易日数
#     新股上市前、退市后的行以 NaN 占位。
#     rolling(20) 是在连续行上操作，不会跨期，但 NaN 会进入窗口。
#     需要区分"结构性 NaN"和"有效期内缺失"（详见模块三）。
#
# [客观评价] 情况B 不代表数据更好，只代表数据提供方做了补全处理。
# 两种情况的 NaN 处理方式不同，必须先判断是哪种，才能正确处理后续计算。

expected_rows = total_stocks * total_dates
fill_rate     = total_rows / expected_rows
print(f"  理论完整行数     : {expected_rows:,}  ({total_stocks} 只 × {total_dates} 天)")
print(f"  实际行数         : {total_rows:,}")
print(f"  面板填充率       : {fill_rate:.2%}")

if fill_rate == 1.0:
    print("  ✓  面板已补全为矩形。NaN 均为行内值缺失，而非缺行。")
    print("  ⚠  上市前/退市后的行以结构性 NaN 占位，后续需用有效区间来区分（见模块三）。")
elif fill_rate > 0.95:
    print("  ⚠  面板接近完整但有少量缺行，建议检查是否需要 reindex 后再做 rolling。")
else:
    print("  ⚠  面板缺口较大，强烈建议先 reindex 补全再做任何时序计算。")
print()

# ── 1.6  每只股票的数据条数分布 ─────────────────────────────────────
# [量化行业判断] 如果所有股票天数相同（= total_dates），
# 说明面板已补全，每只股票在每个交易日都有一行（值可能为 NaN）。
# 如果天数分布差异大，说明面板未补全，新股/退市股覆盖天数不同。
stock_day_count = df.groupby('order_book_id')['date'].count()
print("  每只股票的数据条数分布：")
print(stock_day_count.describe().rename({
    'count': '股票只数', 'mean': '平均覆盖天数', 'std': '标准差',
    'min': '最少天数',   '25%': '25分位',       '50%': '中位数',
    '75%': '75分位',     'max': '最多天数'
}).to_string())
print()

# ── 1.7  每只股票的日期范围和数据条数 ──────────────────────────────
# [量化行业判断] 注意：如果面板已补全（fill_rate=1.0），
# 这里的 min/max 对所有股票都等于样本整体起止日，
# 因为每只股票每天都有一行（值是 NaN）。
# 真正的"有效观测开始日"需要通过模块三的 valid_obs 来判断，
# 而不是用这里的 first_row_date / last_row_date。
stock_date_range = df.groupby('order_book_id')['date'].agg(['min', 'max', 'count'])
stock_date_range.columns = ['first_row_date', 'last_row_date', 'row_count']
print("  各股票日期范围（前5行示例）：")
print(stock_date_range.head(5).to_string())
print()
print("  各股票日期范围（describe）：")
print(stock_date_range.describe().to_string())
print()

# ── 1.8  重复值检查 ──────────────────────────────────────────────────
# [代码工程] (order_book_id, date) 是面板数据的联合主键，不允许重复。
# 重复行会导致：
#   - groupby.apply 的结果行数翻倍（每个 group 内有重复）
#   - rolling 的历史窗口计数错误（同一天被算两次）
#   - merge 产生笛卡尔积（行数爆炸）
# 这类错误不会报异常，会静默产生错误结果，非常危险。
#
# [量化行业判断] 不直接 drop_duplicates，先保存到 dup_df 检查原因。
# 重复可能来自：数据拼接时重叠、数据源有多条成交记录未聚合等，
# 处理方式取决于原因，不能无脑删除第一条或最后一条。
dup_count = df.duplicated(['order_book_id', 'date']).sum()
print(f"  (order_book_id, date) 重复行数 : {dup_count}")

if dup_count > 0:
    dup_df = df[df.duplicated(['order_book_id', 'date'], keep=False)].copy()
    print(f"  ⚠  发现重复主键！已保存到变量 dup_df（{len(dup_df)} 行），请先排查原因再继续。")
    print(dup_df.head(6).to_string())
else:
    print("  ✓  无重复行，联合主键唯一性正常。")
    dup_df = pd.DataFrame()  # 空 DataFrame，方便后续统一引用

【模块一】基础数据体检

  df.shape         : (7100847, 7)
  df.columns       : ['order_book_id', 'date', 'buy_volume', 'buy_value', 'sell_volume', 'sell_value', 'pct_rate']

  字段类型：
    order_book_id         object
    date                  datetime64[ns]
    buy_volume            float64
    buy_value             float64
    sell_volume           float64
    sell_value            float64
    pct_rate              float64

  总行数           : 7,100,847
  独立股票数       : 5,343 只
  独立交易日数     : 1,329 天
  日期范围         : 2020-01-02  →  2025-06-30

  理论完整行数     : 7,100,847  (5343 只 × 1329 天)
  实际行数         : 7,100,847
  面板填充率       : 100.00%
  ✓  面板已补全为矩形。NaN 均为行内值缺失，而非缺行。
  ⚠  上市前/退市后的行以结构性 NaN 占位，后续需用有效区间来区分（见模块三）。

  每只股票的数据条数分布：
股票只数      5343.0
平均覆盖天数    1329.0
标准差          0.0
最少天数      1329.0
25分位      1329.0
中位数       1329.0
75分位      1329.0
最多天数      1329.0

  各股票日期范围（前5行示例）：
              first_row_date last_row_date  row_count
order_book_id                                        
000001.XSHE   